# tssnorm: a tour of the API

tssnorm gives each task its own normalization layers. Each layer's scale becomes
`sigmoid(gamma)`, a value between 0 and 1 per feature. Everything else stays shared.

This notebook walks through the API on a small model. It runs on a CPU in seconds.

1. Wrap a model.
2. Run the tasks.
3. Add task-specific heads.
4. Initialise the scales.
5. Build the optimiser.
6. Read what the scales learned.
7. Push the tasks apart.
8. Start from a pretrained model.
9. Wrap only some norms.
10. Support another norm class.

In [1]:
import torch
from torch import nn

from tssnorm import (SIG_BNS, SIG_LNS, SIG_NORMS, SigLayerNorm, active_task, forward_tasks,
                     get_importance, get_mean_similarity, get_norms, get_optimizer,
                     get_other_param_group, get_param_group, get_similarity, init_norms,
                     make_heads, optim_factory, register_norm, sigmoidify, softmax_loss, taskify,
                     wrap)

torch.manual_seed(0)
TASKS = ["seg", "depth", "normals"]


def encoder():
    return nn.Sequential(
        nn.Conv2d(3, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8), nn.ReLU(),
        nn.Conv2d(8, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(),
        nn.AdaptiveAvgPool2d(1), nn.Flatten(),
    )

## 1. Wrap a model

`wrap` replaces each BatchNorm and LayerNorm with a sigmoid-modulated one, gives each task its
own copy, and initialises the scales. Only the norms are copied. The convolutions stay shared.

Tasks are names, or a number of anonymous tasks: `wrap(model, 4)` makes the tasks `"0"` to `"3"`.

In [2]:
model = encoder()
wrap(model, TASKS)
model

Sequential(
  (0): Conv2d(3, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (1): TSNorm[3 x SigBatchNorm2d(8, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)]
  (2): ReLU()
  (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (4): TSNorm[3 x SigBatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)]
  (5): ReLU()
  (6): AdaptiveAvgPool2d(output_size=1)
  (7): Flatten(start_dim=1, end_dim=-1)
)

## 2. Run the tasks

A wrapped model runs one task at a time. `active_task` sets it for a forward pass.
`forward_tasks` runs every task and returns a dict, or one stacked tensor. Forgetting to set a
task raises an error rather than guessing.

In [3]:
x = torch.randn(2, 3, 16, 16)
model.eval()

with active_task(model, "seg"):
    seg = model(x)
out = forward_tasks(model, x)                      # {task: output}
stacked = forward_tasks(model, x, stack=True)      # [tasks, batch, features]
print(list(out), tuple(stacked.shape))

try:
    model(x)
except RuntimeError as error:
    print("no task set:", str(error).split(";")[0])

['seg', 'depth', 'normals'] (3, 2, 16)
no task set: no active task


## 3. Add task-specific heads

`make_heads` builds one head per task: a sigmoid BatchNorm, then a linear layer. Put the heads
in the model, then wrap it. `wrap` takes the tasks from the heads. The output size can differ
per task.

In [4]:
net = nn.Sequential(encoder(), make_heads(16, {"seg": 21, "depth": 1, "normals": 3}, tasks=TASKS))
wrap(net)
net.eval()
print({task: tuple(y.shape) for task, y in forward_tasks(net, x).items()})

{'seg': (2, 21), 'depth': (2, 1), 'normals': (2, 3)}


## 4. Initialise the scales

`init` sets `gamma`. The default, `const`, starts every scale at `sigmoid(0) = 0.5`, so no task
prefers any feature. `normal` draws a different start per task. `init_norms` re-initialises a
wrapped model.

In [5]:
print("const :", get_importance(net).unique().tolist())
init_norms(net, how="normal")
I = get_importance(net)
print(f"normal: between {I.min():.2f} and {I.max():.2f}")

const : [0.5]
normal: between 0.10 and 0.86


## 5. Build the optimiser

The scales need a higher learning rate than the weights. `optim_factory` puts them in their own
group, at 100 times the base learning rate and without weight decay. For other groupings, pass
`get_optimizer` a list of groups with an `lr_mul` each. Here BatchNorm and LayerNorm scales get
different rates.

In [6]:
optim = optim_factory(net, torch.optim.SGD, lr=1e-2, momentum=0.9, weight_decay=1e-4)
print([(group["lr"], len(group["params"])) for group in optim.param_groups])

optim = get_optimizer([
    {**get_param_group(net, SIG_BNS), "lr_mul": 100},
    {**get_param_group(net, SIG_LNS), "lr_mul": 10},          # empty: this model has no LayerNorm
    get_other_param_group(net, excluding=SIG_NORMS),
], torch.optim.SGD, lr=1e-2, momentum=0.9)
print([(group["lr"], len(group["params"])) for group in optim.param_groups])

[(1.0, 6), (0.01, 11)]
[(1.0, 9), (0.1, 0), (0.01, 8)]


## 6. Read what the scales learned

The scales are a task-feature importance matrix, `[tasks, features]`. `get_importance` reads it,
over the whole model or per layer. `get_similarity` gives the cosine between tasks.
`get_mean_similarity` reduces it to one number, 1 for identical tasks.

In [7]:
print("importance  :", tuple(get_importance(net).shape))
print("per layer   :", [tuple(I.shape) for I in get_importance(net, per_layer=True)])
print("similarity  :\n", get_similarity(net).numpy().round(2))
print("mean        :", round(get_mean_similarity(net), 3))

importance  : (3, 24)
per layer   : [(3, 8), (3, 16)]
similarity  :
 [[1.   0.84 0.86]
 [0.84 1.   0.87]
 [0.86 0.87 1.  ]]
mean        : 0.859


## 7. Push the tasks apart

`softmax_loss` is a regulariser on the scales. It pushes each feature towards fewer tasks. A
lower `tau` pushes harder. `get_norms` selects the norms it acts on, by module name, here the
encoder's second norm. A few steps on the regulariser alone lower the similarity.

In [8]:
norms = get_norms(net, "0.4")
print("regularised:", [name for name, m in net.named_modules() if any(m is n for n in norms)])

optim = optim_factory(net, torch.optim.SGD, lr=1e-2)
before = get_mean_similarity(net)
for _ in range(20):
    optim.zero_grad()
    softmax_loss(norms, tau=0.1).backward()
    optim.step()
print(f"mean similarity {before:.3f} -> {get_mean_similarity(net):.3f}")

regularised: ['0.4']
mean similarity 0.859 -> 0.456


## 8. Start from a pretrained model

Pass `convert` to keep the pretrained scales. `clip` keeps every scale between 0 and 1, and
replaces the others with 0.95 or 0.05. `scale` adds a fixed factor `k` per layer, so it also
keeps scales above 1. A `+bias` suffix keeps the shift too. With `scale+bias`, a model whose
scales are all positive computes exactly what it did before. A negative scale cannot be kept.

In [9]:
def pretrained():
    torch.manual_seed(1)
    m = nn.Sequential(nn.Conv2d(3, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8))
    m[1].weight.data = torch.tensor([0.001, 0.2, 0.5, 0.9, 0.99, 1.3, 2.0, 0.6])
    m[1].bias.data = torch.randn(8)
    return m.eval()


reference = pretrained()(x)
for how in ["clip+bias", "scale+bias"]:
    m = wrap(pretrained(), TASKS, convert=how)
    with active_task(m, "seg"):
        same = torch.allclose(m(x), reference, atol=1e-5)
    print(f"{how:10s} same output: {same}")

clip+bias  same output: False
scale+bias same output: True


## 9. Wrap only some norms

`wrap` runs three steps: `sigmoidify` modulates the norms, `taskify` copies them per task, and
`init_norms` initialises them. `sigmoidify` and `taskify` take `norms`, the layer types to act
on, so parts of a model can be treated differently. Here the LayerNorms become task-specific but stay plain, and
only the BatchNorm is modulated.

In [10]:
mixed = nn.Sequential(nn.Linear(8, 8), nn.LayerNorm(8), nn.Linear(8, 8), nn.BatchNorm1d(8))
taskify(mixed, TASKS, norms=nn.LayerNorm)          # task-specific, not modulated
wrap(mixed, TASKS, norms=nn.BatchNorm1d)           # task-specific and modulated
print([type(norm["seg"]).__name__ for norm in get_norms(mixed)])

['LayerNorm', 'SigBatchNorm1d']


## 10. Support another norm class

`wrap` refuses norm classes it does not know, such as timm's `LayerNorm` subclass. A subclass
may change the forward pass, so replacing it is not assumed safe. Register one that behaves
like its base class.

In [11]:
import timm.layers

block = nn.Sequential(nn.Linear(8, 8), timm.layers.LayerNorm(8))
try:
    wrap(block, TASKS)
except TypeError as error:
    print("refused:", str(error).split(":")[0])

register_norm(timm.layers.LayerNorm, SigLayerNorm)
wrap(block, TASKS)
print("wrapped:", type(block[1]["seg"]).__name__)

refused: cannot modulate timm.layers.norm.LayerNorm at '1'
wrapped: SigLayerNorm
